# Veri Temizleme ve Hazırlama

Bu notebook, TÜİK MEDAS ve İstatistiklerle Yaşlılar 2025 bülteninden indirilen ham dosyaları okur, temizler, TÜİK’in açıkladığı rakamlarla doğrular ve analizde kullanılacak veri/islenmis/ana_tablo.csv dosyasını üretir.

Kullanılan kütüphaneleri yükleme.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Dosya yapısının incelenmesi

Göç dosyasını başlık varsaymadan ham haliyle okuma ve ilk satırlarını inceleme.

In [2]:
yol = "../veri/ham/goc_yas_2008_2014.csv"
ham = pd.read_csv(yol, sep="|", header=None, dtype=str, encoding="utf-8-sig")
ham.iloc[:7, :6]

,0,1,2,3,4,5
0,NaN,NaN,Sütunlar,NaN,NaN,NaN
1,Satırlar,NaN,Bölgelerin Aldığı Göç Bilgileri,NaN,NaN,NaN
2,NaN,NaN,0-4,10-14,15-19,20-24
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
5,2008,Adana-1,3442.0,3545.0,3636.0,7464.0
6,NaN,Adıyaman-2,1066.0,794.0,1003.0,1977.0


Gösterge ve yaş grubu satırlarından kısa sütun başlıkları oluşturma.

In [3]:
gosterge = ham.iloc[1, 2:].ffill()
yas = ham.iloc[2, 2:]
basliklar = gosterge.str.contains("Aldığı").map({True: "aldigi", False: "verdigi"}) + "_" + yas

Veri satırlarını ayırma, boş yıl hücrelerini doldurma ve sütunlara isim verme.

In [4]:
veri = ham.iloc[5:, :].copy()
veri[0] = veri[0].ffill()                  
veri = veri[veri[1].notna()]               
veri.columns = ["yil", "il"] + list(basliklar)
veri = veri.loc[:, veri.columns.notna()]   
print(veri.shape)

(567, 30)


İl sütununu il adı ve plaka kodu olarak ikiye ayırma.

In [5]:
veri[["il_adi", "il_kodu"]] = veri["il"].str.rsplit("-", n=1, expand=True)
veri["il_kodu"] = veri["il_kodu"].astype(int)

Yıl sütununu tam sayıya çevirme.

In [6]:
veri["yil"] = veri["yil"].astype(float).astype(int)

Göç sütunlarını metinden tam sayıya çevirme.

In [7]:
sayisal = veri.filter(regex="^(aldigi|verdigi)_").columns
veri[sayisal] = veri[sayisal].apply(pd.to_numeric).astype(int)

Artık gerekmeyen il sütununu silme.

In [8]:
veri = veri.drop(columns="il")

Sütun türlerini kontrol etme.

In [9]:
veri.dtypes.value_counts()

int64    30
str       1
Name: count, dtype: int64

İstanbul'un 2008 yılında aldığı ve verdiği göçü TÜİK rakamlarıyla karşılaştırma.

In [10]:
ist = veri[(veri.yil == 2008) & (veri.il_adi == "İstanbul")]
ist.filter(like="aldigi_").sum(axis=1), ist.filter(like="verdigi_").sum(axis=1)

(44    374868
 dtype: int64,
 44    348193
 dtype: int64)

## 2. MEDAS okuma fonksiyonu

Gösterge adına göre sütun öneki (aldigi, verdigi, nufus) belirleyen yardımcı fonksiyonu tanımlama.

In [11]:
def onek_bul(metin):
    if "Aldığı" in metin:
     return "aldigi"
    elif "Verdiği" in metin:
     return "verdigi"
    elif "Nüfus" in metin:
     return "nufus"

MEDAS'tan indirilen, yıl ve illerin satırlarda yer aldığı dosyaları okuyup temizleyen fonksiyonu tanımlama.

In [ ]:
def medas_oku(yol):
    ham = pd.read_csv(yol, sep="|", header=None, dtype=str, encoding="utf-8-sig")
    gosterge = ham.iloc[1, 2:].ffill()
    yas = ham.iloc[2, 2:]
    basliklar = gosterge.map(onek_bul) + "_" + yas
    veri = ham.iloc[5:, :].copy()
    veri[0] = veri[0].ffill()
    veri = veri[veri[1].notna()]
    veri.columns = ["yil", "il"] + list(basliklar)
    veri = veri.loc[:, veri.columns.notna()]
    veri[["il_adi", "il_kodu"]] = veri["il"].str.rsplit("-", n=1, expand=True)
    veri["il_kodu"] = veri["il_kodu"].astype(int)
    veri["yil"] = veri["yil"].astype(float).astype(int)
    sayisal = veri.columns.drop(["yil", "il", "il_adi", "il_kodu"])
    veri[sayisal] = veri[sayisal].apply(pd.to_numeric).astype(int)
    veri = veri.drop(columns="il")
    return veri

## 3. Göç verisi

Üç göç dosyasını okuma ve tek tabloda birleştirme.

In [13]:
goc_dosyalari = ["goc_yas_2008_2014.csv", "goc_yas_2015_2021.csv", "goc_yas_2022_2025.csv"]
goc_yas = pd.concat([medas_oku("../veri/ham/" + d) for d in goc_dosyalari], ignore_index=True)

Göç tablosunun boyutunu, her yıldaki il sayısını ve 2025 toplamını kontrol etme.

In [14]:
print(goc_yas.shape)
print(goc_yas.groupby("yil")["il_kodu"].nunique().unique())
print(goc_yas[goc_yas.yil == 2025].filter(like="aldigi_").sum().sum())

(1458, 31)
[81]
2475019


#### Göç matrisiyle doğrulama

İller arası göç matrisini okuyup her satırda veren il, alan il ve göç eden kişi sayısının yer aldığı uzun formata çeviren fonksiyonu tanımlama.

In [15]:
def matris_oku(yol):
    ham = pd.read_csv(yol, sep="|", header=None, dtype=str, encoding="utf-8-sig")
    alan_iller = ham.iloc[2, 2:]
    veri = ham.iloc[5:, :].copy()
    veri[0] = veri[0].ffill()
    veri = veri[veri[1].notna()]
    veri.columns = ["yil", "veren_il"] + list(alan_iller)
    veri = veri.loc[:, veri.columns.notna()]
    uzun = veri.melt(id_vars=["yil", "veren_il"], var_name="alan_il", value_name="kisi")
    uzun["yil"] = uzun["yil"].astype(float).astype(int)
    uzun["il_kodu"] = uzun["veren_il"].str.rsplit("-", n=1).str[1].astype(int)
    uzun["kisi"] = pd.to_numeric(uzun["kisi"])
    return uzun

Üç göç matrisi dosyasını okuma, tek tabloda birleştirme ve boş hücreleri listeleme.

In [16]:
matris_dosyalari = ["goc_matrisi_2008_2014.csv", "goc_matrisi_2015_2021.csv", "goc_matrisi_2022_2025.csv"]
goc_matrisi = pd.concat([matris_oku("../veri/ham/" + d) for d in matris_dosyalari], ignore_index=True)
print(goc_matrisi.shape, goc_matrisi["kisi"].isna().sum())
goc_matrisi[goc_matrisi["kisi"].isna()][["yil", "veren_il", "alan_il"]]

(118098, 5) 5


,yil,veren_il,alan_il
59015,2015,Kilis-79,Göç Alan:Çorum
72976,2019,Uşak-64,Göç Alan:Kilis
73070,2021,Ardahan-75,Göç Alan:Kilis
73131,2021,Sinop-57,Göç Alan:Kilis
85097,2015,Kilis-79,Göç Alan:Sinop


Boş hücreleri sıfır kabul etme ve her il ve yıl için matristen hesaplanan verdiği göçü, yaş gruplarına göre göç verisindeki toplamla karşılaştırma.

In [17]:
goc_matrisi["kisi"] = goc_matrisi["kisi"].fillna(0).astype(int)
matris_verdigi = goc_matrisi.groupby(["yil", "il_kodu"])["kisi"].sum()
yas_verdigi = goc_yas.set_index(["yil", "il_kodu"]).filter(like="verdigi_").sum(axis=1)
fark = (matris_verdigi - yas_verdigi).abs()
print(len(fark), fark.isna().sum(), fark.max())

1458 0 0


## 4. Nüfus verisi

Üç nüfus dosyasını okuma ve tek tabloda birleştirme.

In [18]:
nufus_dosyalari = ["nufus_yas_2008_2014.csv", "nufus_yas_2015_2021.csv", "nufus_yas_2022_2025.csv"]
nufus_yas = pd.concat([medas_oku("../veri/ham/" + d) for d in nufus_dosyalari], ignore_index=True)

Nüfus tablosunun boyutunu, 2025 toplam nüfusunu ve 65 yaş üstü nüfusunu kontrol etme.

In [19]:
n25 = nufus_yas[nufus_yas.yil == 2025]
print(nufus_yas.shape)
print(n25.filter(like="nufus_").sum().sum())
print(n25[["nufus_65-69", "nufus_70-74", "nufus_75-79", "nufus_80-84", "nufus_85-89", "nufus_90+"]].sum().sum())

(1458, 22)
86092168
9583059


## 5. Sağlık ve yaşam beklentisi verisi


### Bölüm 5.1: Geniş formatlı dosyaları okuma fonksiyonu

İllerin sütunlarda yer aldığı geniş formatlı MEDAS dosyalarını okuyup uzun formata çeviren fonksiyonu tanımlama.

In [20]:
def medas_genis_oku(yol):
    ham = pd.read_csv(yol, sep="|", header=None, dtype=str, encoding="utf-8-sig")
 
   
    iller = ham.iloc[1, 3:]
 
    
    veri = ham.iloc[4:, :].copy()
    veri[[0, 1]] = veri[[0, 1]].ffill()
    veri = veri[veri[2].notna()]                      
    veri.columns = ["gosterge", "kirilim", "yil"] + list(iller)
    veri = veri.loc[:, veri.columns.notna()]          
 
    uzun = veri.melt(id_vars=["gosterge", "kirilim", "yil"], var_name="il", value_name="deger")
 

    uzun[["il_adi", "il_kodu"]] = uzun["il"].str.rsplit("-", n=1, expand=True)
    uzun["il_kodu"] = uzun["il_kodu"].astype(int)
    uzun["yil"] = uzun["yil"].astype(float).astype(int)
    uzun["deger"] = pd.to_numeric(uzun["deger"])      
    uzun = uzun.drop(columns="il")
    return uzun

### Bölüm 5.2: Verileri okuma

Sağlık personeli, yatak oranı, hekim oranı ve yaşam beklentisi dosyalarını okuma.

In [21]:
saglik_personel = medas_genis_oku("../veri/ham/saglik_personel_2022_2024.csv")
saglik_yatak = medas_genis_oku("../veri/ham/saglik_yatak_orani_2022_2024.csv")
saglik_hekim_orani = medas_genis_oku("../veri/ham/saglik_hekim_orani_2022_2024.csv")
yasam_beklentisi = medas_genis_oku("../veri/ham/yasam_beklentisi_2023.csv")

### Bölüm 5.3: Kontroller

#### Kontrol 1: satır sayıları

Dört tablonun satır sayılarını ve boş değer sayılarını kontrol etme.

In [22]:
for ad, tablo in [("personel", saglik_personel), ("yatak", saglik_yatak),
                  ("hekim oranı", saglik_hekim_orani), ("yaşam beklentisi", yasam_beklentisi)]:
    print(ad, tablo.shape, "| boş değer:", tablo["deger"].isna().sum())

personel (972, 6) | boş değer: 0
yatak (243, 6) | boş değer: 0
hekim oranı (243, 6) | boş değer: 0
yaşam beklentisi (162, 6) | boş değer: 0


#### Kontrol 2: Ankara 2024 hekim sayıları

Ankara'nın 2024 hekim sayılarını unvanlara göre gösterme.

In [23]:
saglik_personel[(saglik_personel.yil == 2024) & (saglik_personel.il_adi == "Ankara")]

,gosterge,kirilim,yil,deger,il_adi,il_kodu
62,Sağlık Personeli Sayısı,1. (Uzman Hekim),2024,12064.0,Ankara,6
65,Sağlık Personeli Sayısı,2. (Pratisyen Hekim),2024,3448.0,Ankara,6
68,Sağlık Personeli Sayısı,3. (Asistan Hekim),2024,11286.0,Ankara,6
71,Sağlık Personeli Sayısı,4. (Diş Hekimi),2024,5158.0,Ankara,6


#### Kontrol 3: Türkiye 2024 toplam hekim sayısı (diş hekimleri hariç)

Diş hekimleri hariç Türkiye'nin 2024 toplam hekim sayısını hesaplama.

In [24]:
hekimler = saglik_personel[saglik_personel.kirilim != "4. (Diş Hekimi)"]
print(hekimler[hekimler.yil == 2024]["deger"].sum())

221133.0


## 6. TÜİK yaşlı tabloları

### Bölüm 6.1: Dosyayı ham haliyle okumak

Yalnız yaşlıların çocuklarına uzaklığını gösteren Excel tablosunu ham haliyle okuma.

In [25]:
ham_yakinlik = pd.read_excel("../veri/ham/yasli_cocuk_yakinlik_2025.xls", header=None, engine="calamine")
print(ham_yakinlik.shape)
ham_yakinlik.iloc[4:9, :4]

(102, 11)


,0,1,2,3
4,İl\nProvince,NaN,NaN,Aynı ilde çocuğu olan yaşlı fertler\nElderly p...
5,NaN,Toplam\nTotal,Çocuğu yok/\n15 ve daha\nyukarı yaşta\nçocuğu ...,Aynı binada\nçocuğu olan\nyaşlı fertler\nElder...
6,Toplam-Total,100,9.690792,10.097272
7,Adana,100,10.677194,13.716982
8,Adıyaman,100,6.798643,12.058824


### Bölüm 6.2: Veri satırlarını ayırmak ve sütunlara isim vermek

81 ilin veri satırlarını ayırma, sütunlara isim verme ve il adlarındaki boşlukları temizleme.

In [26]:
yakinlik = ham_yakinlik.iloc[7:88].copy()
yakinlik.columns = ["il_adi", "toplam", "cocugu_yok", "ayni_bina", "ayni_sokak",
                    "ayni_mahalle", "ayni_ilce", "farkli_ilce", "farkli_il",
                    "yurtdisi", "bilinmeyen"]
yakinlik["il_adi"] = yakinlik["il_adi"].str.strip()
print(yakinlik.shape)
yakinlik.iloc[[0, -1], :4]

(81, 11)


,il_adi,toplam,cocugu_yok,ayni_bina
7,Adana,100,10.677194,13.716982
87,Düzce,100,9.18548,10.734838


### Bölüm 6.3: “-” işaretlerini sıfıra çevirmek ve sayıya dönüştürmek

Tablodaki "-" işaretlerinin hangi sütunlarda kaç tane olduğunu sayma.

In [27]:
eksi = (yakinlik == "-").sum()
eksi[eksi > 0]

yurtdisi       1
bilinmeyen    14
dtype: int64

"-" işaretlerini sıfıra çevirme ve bütün sütunları sayıya dönüştürme.

In [28]:
sayisal = yakinlik.columns.drop("il_adi")
yakinlik[sayisal] = yakinlik[sayisal].replace("-", "0").apply(pd.to_numeric)
yakinlik.dtypes.value_counts()

float64    9
str        1
int64      1
Name: count, dtype: int64

Satır toplamlarının 100 olup olmadığını kontrol etme ve Çankırı değerini TÜİK rakamıyla karşılaştırma.

In [29]:
satir_toplami = yakinlik[sayisal].drop(columns="toplam").sum(axis=1)
print(satir_toplami.min(), satir_toplami.max())
yakinlik.loc[yakinlik.il_adi == "Çankırı", ["il_adi", "cocugu_yok", "farkli_il"]]

99.99999999999997 100.00000000000003


,il_adi,cocugu_yok,farkli_il
24,Çankırı,4.7875,40.8875


### Bölüm 6.4: Plaka kodunu eklemek

Nüfus tablosundan il adı ve plaka kodu eşleştirme tablosu oluşturma.

In [30]:
il_kodlari = nufus_yas[["il_adi", "il_kodu"]].drop_duplicates()
print(len(il_kodlari))

81


Plaka kodlarını il adı üzerinden tabloya ekleme ve bütün illerin eşleşip eşleşmediğini kontrol etme.

In [31]:
yakinlik = yakinlik.merge(il_kodlari, on="il_adi", how="left",
                          indicator=True, validate="one_to_one")
yakinlik["_merge"].value_counts()

_merge
both          81
left_only      0
right_only     0
Name: count, dtype: int64

Kontrol sütununu silme, sütunları düzenleme ve tabloyu son kez kontrol etme.

In [32]:
yakinlik = yakinlik.drop(columns="_merge")
ilk = ["il_kodu", "il_adi"]
yakinlik = yakinlik[ilk + [c for c in yakinlik.columns if c not in ilk]]
print(yakinlik.shape, yakinlik["il_kodu"].is_unique)
yakinlik.iloc[[0, -1], :4]

(81, 12) True


,il_kodu,il_adi,toplam,cocugu_yok
0,1,Adana,100,10.677194
80,81,Düzce,100,9.185480


### Bölüm 6.5: İkinci Excel dosyası (tek kişilik yaşlı haneler)

Tek kişilik yaşlı hanelere ait Excel tablosunu ham haliyle okuma.

In [33]:
ham_hanehalki = pd.read_excel("../veri/ham/yasli_hanehalki_2025.xls", header=None, engine="calamine")
print(ham_hanehalki.shape)
ham_hanehalki.iloc[4:8]

(92, 7)


,0,1,2,3,4,5,6
4,İl\nProvince,En az bir yaşlı\nfert bulunan\nhanehalklarının...,Tek kişilik yaşlı\nhanehalklarının\ntoplam han...,Tek kişilik yaşlı\nhanehalklarının\ntoplam tek...,NaN,Cinsiyete göre tek kişilik yaşlı\nhanehalkı or...,NaN
5,NaN,NaN,NaN,NaN,NaN,Erkek\nMale,Kadın\nFemale
6,Toplam-Total,26.119852,6.807436,33.249851,NaN,26.485329,73.514671
7,Adana,26.45088,6.169063,33.306482,NaN,25.947266,74.052734


Veri satırlarını ayırma, boş sütunu atma ve sütunlara isim verme.

In [34]:
hanehalki = ham_hanehalki.iloc[7:88].drop(columns=4).copy()
hanehalki.columns = ["il_adi", "yasli_hane_orani", "tek_yasli_hane_orani",
                     "tek_yasli_tek_kisilik_icinde", "tek_yasli_erkek", "tek_yasli_kadin"]
hanehalki["il_adi"] = hanehalki["il_adi"].str.strip()
print(hanehalki.shape, (hanehalki == "-").sum().sum())

(81, 6) 0


Değerleri sayıya çevirme, plaka kodunu ekleme ve sütunları düzenleme.

In [35]:
sayisal = hanehalki.columns.drop("il_adi")
hanehalki[sayisal] = hanehalki[sayisal].apply(pd.to_numeric)
hanehalki = hanehalki.merge(il_kodlari, on="il_adi", how="left",
                            indicator=True, validate="one_to_one")
print(hanehalki["_merge"].value_counts())
hanehalki = hanehalki.drop(columns="_merge")
ilk = ["il_kodu", "il_adi"]
hanehalki = hanehalki[ilk + [c for c in hanehalki.columns if c not in ilk]]

_merge
both          81
left_only      0
right_only     0
Name: count, dtype: int64


Cinsiyet oranlarının toplamını kontrol etme ve Balıkesir ile Hakkari değerlerini TÜİK rakamlarıyla karşılaştırma.

In [36]:
cinsiyet_toplami = hanehalki["tek_yasli_erkek"] + hanehalki["tek_yasli_kadin"]
print(hanehalki.shape, hanehalki["il_kodu"].is_unique)
print(cinsiyet_toplami.min(), cinsiyet_toplami.max())
b = hanehalki.set_index("il_adi")
(b["tek_yasli_hane_orani"] / b["yasli_hane_orani"] * 100).loc[["Balıkesir", "Hakkari"]].round(1)

(81, 7) True
99.99999999999999 100.00000000000001


il_adi
Balıkesir    34.3
Hakkari       8.3
dtype: float64

## 7. Ana tablonun oluşturulması

### Bölüm 7.1: Genç net göç

Her il ve yıl için 15-34 yaş grubunun aldığı, verdiği ve net göçünü hesaplama.

In [37]:
genc = ["15-19", "20-24", "25-29", "30-34"]
goc_yas["genc_aldigi"] = goc_yas[["aldigi_" + y for y in genc]].sum(axis=1)
goc_yas["genc_verdigi"] = goc_yas[["verdigi_" + y for y in genc]].sum(axis=1)
goc_yas["genc_net_goc"] = goc_yas["genc_aldigi"] - goc_yas["genc_verdigi"]

Her yıl illerin net göç toplamının sıfır olup olmadığını kontrol etme.

In [38]:
goc_yas.groupby("yil")["genc_net_goc"].sum().unique()

array([0])

2008-2022 dönemindeki genç göçünü her il için toplama.

In [39]:
donem = goc_yas[goc_yas.yil.between(2008, 2022)]
goc_il = (donem.groupby(["il_kodu", "il_adi"])[["genc_aldigi", "genc_verdigi", "genc_net_goc"]]
          .sum().reset_index())
print(goc_il.shape, donem.yil.nunique())
goc_il.sort_values("genc_net_goc").iloc[[0, 1, 2, -3, -2, -1]]

(81, 5) 15


,il_kodu,il_adi,genc_aldigi,genc_verdigi,genc_net_goc
64,65,Van,266835,376563,-109728
0,1,Adana,422556,528702,-106146
3,4,Ağrı,140719,238865,-98146
34,35,İzmir,983877,827720,156157
5,6,Ankara,1552625,1259186,293439
33,34,İstanbul,3501627,2717476,784151


### Bölüm 7.2: Nüfus değişkenleri ve göç hızı

Her il ve yıl için toplam, genç ve yaşlı nüfusu hesaplama.

In [40]:
yasli = ["65-69", "70-74", "75-79", "80-84", "85-89", "90+"]
nufus_yas["toplam_nufus"] = nufus_yas.filter(regex="^nufus_").sum(axis=1)
nufus_yas["genc_nufus"] = nufus_yas[["nufus_" + y for y in genc]].sum(axis=1)
nufus_yas["yasli_nufus"] = nufus_yas[["nufus_" + y for y in yasli]].sum(axis=1)
nufus_yas[nufus_yas.yil == 2025][["toplam_nufus", "yasli_nufus"]].sum()

toplam_nufus    86092168
yasli_nufus      9583059
dtype: int64

İl düzeyinde 2025 nüfusu, ortalama genç nüfus ve yaşlı oranından oluşan tabloyu oluşturma.

In [41]:
ort_genc = (nufus_yas[nufus_yas.yil.between(2008, 2022)]
            .groupby("il_kodu")["genc_nufus"].mean().rename("ort_genc_nufus"))
n2025 = nufus_yas[nufus_yas.yil == 2025].set_index("il_kodu")[["toplam_nufus", "yasli_nufus"]]
nufus_il = n2025.join(ort_genc).reset_index()
nufus_il["yasli_orani"] = nufus_il["yasli_nufus"] / nufus_il["toplam_nufus"] * 100
print(nufus_il.shape)

(81, 5)


Genç net göç hızını binde olarak hesaplama.

In [42]:
goc_nufus = goc_il.merge(nufus_il, on="il_kodu", validate="one_to_one")
goc_nufus["genc_goc_hizi"] = goc_nufus["genc_net_goc"] / goc_nufus["ort_genc_nufus"] * 1000
print(goc_nufus.shape)
goc_nufus.sort_values("genc_goc_hizi")[["il_adi", "genc_net_goc", "genc_goc_hizi"]].iloc[[0, 1, 2, -3, -2, -1]]

(81, 10)


,il_adi,genc_net_goc,genc_goc_hizi
3,Ağrı,-98146,-494.689827
48,Muş,-71642,-481.243406
74,Ardahan,-15715,-474.135503
76,Yalova,17675,248.817267
40,Kocaeli,152072,257.775891
58,Tekirdağ,118353,390.716662


#### Sağlamlık kontrolü için 25-34 yaş göç hızı

Sağlamlık kontrolü için yalnızca 25-34 yaş grubunu kapsayan göç hızını hesaplama.

In [43]:
genc2 = ["25-29", "30-34"]
goc_yas["genc2_net_goc"] = (goc_yas[["aldigi_" + y for y in genc2]].sum(axis=1)
                            - goc_yas[["verdigi_" + y for y in genc2]].sum(axis=1))
nufus_yas["genc2_nufus"] = nufus_yas[["nufus_" + y for y in genc2]].sum(axis=1)
net = goc_yas[goc_yas.yil.between(2008, 2022)].groupby("il_kodu")["genc2_net_goc"].sum()
ort = nufus_yas[nufus_yas.yil.between(2008, 2022)].groupby("il_kodu")["genc2_nufus"].mean()
goc_nufus["genc2_goc_hizi"] = goc_nufus["il_kodu"].map(net / ort * 1000)
print(goc_yas.groupby("yil")["genc2_net_goc"].sum().unique(), goc_nufus["genc2_goc_hizi"].isna().sum())

[0] 0


### Bölüm 7.3: Sağlık değişkenleri

Diş hekimleri hariç her il ve yıl için toplam hekim sayısını hesaplama.

In [44]:
hekimler = saglik_personel[saglik_personel.kirilim != "4. (Diş Hekimi)"]
hekim_il = hekimler.groupby(["il_kodu", "yil"])["deger"].sum().rename("hekim_sayisi").reset_index()
print(hekim_il.shape)

(243, 3)


Bin kişi başına hekim oranını hesaplama ve TÜİK'in yayımladığı oranlarla karşılaştırma.

In [45]:
nufus_yillik = nufus_yas[["il_kodu", "yil", "toplam_nufus"]]
hekim_il = hekim_il.merge(nufus_yillik, on=["il_kodu", "yil"], how="left", validate="one_to_one")
hekim_il["hekim_orani"] = hekim_il["hekim_sayisi"] / hekim_il["toplam_nufus"] * 1000
resmi = saglik_hekim_orani[["il_kodu", "yil", "deger"]].rename(columns={"deger": "resmi_oran"})
kontrol = hekim_il.merge(resmi, on=["il_kodu", "yil"], validate="one_to_one")
print(len(kontrol), (kontrol["hekim_orani"].round(1) == kontrol["resmi_oran"]).sum())

243 243


2024 yılı için hekim ve yatak oranlarından oluşan il düzeyinde sağlık tablosunu oluşturma.

In [46]:
saglik_il = hekim_il[hekim_il.yil == 2024][["il_kodu", "hekim_orani"]]
yatak_2024 = (saglik_yatak[saglik_yatak.yil == 2024][["il_kodu", "deger"]]
              .rename(columns={"deger": "yatak_orani"}))
saglik_il = saglik_il.merge(yatak_2024, on="il_kodu", validate="one_to_one")
print(saglik_il.shape)
saglik_il.drop(columns="il_kodu").describe().round(2)

(81, 3)


,hekim_orani,yatak_orani
count,81.00,81.00
mean,2.12,299.70
std,0.64,83.18
min,1.20,130.00
25%,1.67,249.00
50%,2.01,277.00
75%,2.51,327.00
max,4.57,532.00


### Bölüm 7.4: Yaşam beklentisi farkı

Erkek ve kadın yaşam beklentisini yan yana getirme ve aradaki farkı hesaplama.

In [47]:
yasam_il = (yasam_beklentisi.pivot(index="il_kodu", columns="kirilim", values="deger")
            .rename(columns={"Erkek": "yasam_erkek", "Kadın": "yasam_kadin"})
            .reset_index())
yasam_il.columns.name = None
yasam_il["yasam_farki"] = yasam_il["yasam_kadin"] - yasam_il["yasam_erkek"]
print(yasam_il.shape)
yasam_il.drop(columns="il_kodu").describe().round(2)

(81, 4)


,yasam_erkek,yasam_kadin,yasam_farki
count,81.00,81.00,81.00
mean,75.34,80.78,5.44
std,0.86,0.94,0.57
min,73.20,78.70,4.10
25%,74.80,80.20,5.10
50%,75.30,80.70,5.50
75%,76.00,81.30,5.70
max,78.10,83.60,7.40


### Bölüm 7.5: Ana tabloyu birleştirmek ve kaydetmek

Beş tabloyu plaka kodu üzerinden tek bir ana tabloda birleştirme.

In [48]:
ana_tablo = (yakinlik.drop(columns="toplam")
             .merge(hanehalki.drop(columns="il_adi"), on="il_kodu", validate="one_to_one")
             .merge(goc_nufus.drop(columns="il_adi"), on="il_kodu", validate="one_to_one")
             .merge(saglik_il, on="il_kodu", validate="one_to_one")
             .merge(yasam_il, on="il_kodu", validate="one_to_one"))
print(ana_tablo.shape, ana_tablo.isna().sum().sum())

(81, 30) 0


Tek kişilik yaşlı hane payını ve deprem bölgesindeki illeri gösteren değişkenleri ekleme.

In [49]:
ana_tablo["tek_yasli_payi"] = ana_tablo["tek_yasli_hane_orani"] / ana_tablo["yasli_hane_orani"] * 100
deprem_illeri = [1, 2, 21, 23, 27, 31, 44, 46, 63, 79, 80]
ana_tablo["deprem_ili"] = ana_tablo["il_kodu"].isin(deprem_illeri).astype(int)
ana_tablo.loc[ana_tablo["deprem_ili"] == 1, "il_adi"].tolist()

['Adana',
 'Adıyaman',
 'Diyarbakır',
 'Elazığ',
 'Gaziantep',
 'Hatay',
 'Malatya',
 'Kahramanmaraş',
 'Şanlıurfa',
 'Kilis',
 'Osmaniye']

Ana tabloyu son kez kontrol etme, kaydetme ve kaydedilen dosyayı tekrar okuyarak doğrulama.

In [50]:
print(ana_tablo.shape, ana_tablo["il_kodu"].is_unique, ana_tablo.isna().sum().sum())
Path("../veri/islenmis").mkdir(parents=True, exist_ok=True)
ana_tablo.to_csv("../veri/islenmis/ana_tablo.csv", index=False, encoding="utf-8-sig")
geri_okunan = pd.read_csv("../veri/islenmis/ana_tablo.csv", encoding="utf-8-sig")
print(geri_okunan.shape)

(81, 32) True 0
(81, 32)
